In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import Normalize 
from matplotlib.cm import ScalarMappable
from matplotlib.colors import TwoSlopeNorm

In [6]:
#raw data
df = pd.read_csv(r"/home/lero/idrive/cmac/DDMAP/Stability studies/Stability_dataset_August_update.csv", na_values='nan')
df['Average days stable']=(df['Average days stable']>=2160).astype(int)
#df

In [17]:
#model data
import pickle 

def mean_and_round(series):
    return series.mean().round()

with open('/home/lero/idrive/cmac/DDMAP/Stability studies/Model_results/Classifier/July_api_no/Classifiers_results_dictionary.pkl', 'rb') as f:
    results = pickle.load(f)
    
model_data = pd.DataFrame(results['Random Forest Classifier']['predictions'], columns=['model_stability'])

combined_df = pd.concat([model_data, df], axis = 1)

columns = ['API', 'Polymer', 'condition', 'Drug loading (wt%)', 'model_stability', 'Average days stable']
results_df = combined_df[columns]

pure_df = results_df[results_df['Polymer']=='Pure']
grouped_pure = pure_df.groupby(['API', 'condition'])
rounded_pure = grouped_pure[['Average days stable', 'model_stability']].mean().round().reset_index()
rounded_pure['Polymer'] = 'Pure'
rounded_pure['Drug loading (wt%)'] = 100

dropped_pure = results_df[results_df['Polymer'] !='Pure']
classifier_df= pd.concat([dropped_pure, rounded_pure], ignore_index=True)
#classifier_df['Average days stable']=(classifier_df['Average days stable']>=2160).astype(int)
classifier_df['stability difference'] = (classifier_df['Average days stable']-classifier_df['model_stability'])

#classifier_df.to_csv('/projects/cp/se_users/ksrn200/Classifier/GroupKFold/df_check.csv', index=False)
classifier_df

,API,Polymer,condition,Drug loading (wt%),model_stability,Average days stable,stability difference
0,Acetaminophen,Soluplus,40_C_75_RH,60,0.0,1.0,1.0
1,Acetaminophen,Soluplus,40_C_75_RH,50,0.0,1.0,1.0
2,Acetaminophen,Soluplus,40_C_75_RH,40,0.0,1.0,1.0
3,Acetaminophen,Soluplus,40_C_75_RH,30,1.0,1.0,0.0
4,Acetaminophen,Soluplus,40_C_75_RH,20,1.0,1.0,0.0
...,...,...,...,...,...,...,...
3931,Sulfamerazine,Pure,40_C_0_RH,100,0.0,0.0,0.0
3932,Sulfamerazine,Pure,40_C_75_RH,100,0.0,0.0,0.0
3933,Tinidazole,Pure,30_C_30_RH,100,1.0,0.0,-1.0
3934,Tinidazole,Pure,40_C_0_RH,100,1.0,1.0,0.0


In [18]:
#plot heatmaps stability
import matplotlib.patches as patches
import matplotlib.patches as mpatches

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import Normalize
import os

norm = Normalize(vmin=0, vmax=1.0)

for condition, condition_group in classifier_df.groupby('condition'):
    apis = condition_group['API'].unique
    
    figure, axes = plt.subplots(4,8, figsize=(20,15), dpi=700, sharey=True)
    axes=axes.flatten()
    #cbar_ax = figure.add_axes([0.91, 0.3, 0.03, 0.4,])

    for i, (api, group) in enumerate(condition_group.groupby('API')):
        if i>=32:
            break
        ax=axes[i]
        heatmap_data = group.pivot_table(
        index='Drug loading (wt%)', 
        columns='Polymer',
        values='model_stability',
    )
        
        heatmap_data=heatmap_data.sort_index(ascending=False)
        #move pure to last column
        if 'Pure' in heatmap_data.columns:
            cols = [col for col in heatmap_data.columns if col != 'Pure'] + ['Pure']
            heatmap_data = heatmap_data[cols]
            sns.heatmap(heatmap_data, ax=ax, cbar=False, cmap='gray', linewidth=0.5, linecolor='black', norm=norm)
            ax.set_title(f'{api}')
            ax.set_xlabel('Polymer')
            ax.set_ylabel('Drug loading (wt%)')
        
        
        # Draw green outlines where 'Average days stable' == 1 in this API
        truth_data = group[group['Average days stable'] == 0]
        for _, truth_row in truth_data.iterrows():
            row_idx = heatmap_data.index.get_loc(truth_row['Drug loading (wt%)'])
            col_idx = heatmap_data.columns.get_loc(truth_row['Polymer'])
            rect = patches.Rectangle((col_idx, row_idx), 1, 1, linewidth=2, edgecolor='g', facecolor='none')
            ax.add_patch(rect)

    # Add a legend/key for the green rectangle
    green_patch = mpatches.Patch(facecolor='none', edgecolor='g', linewidth=2, label='True unstable ASD')
    black_patch = mpatches.Patch(facecolor='k', edgecolor='k', linewidth=2, label='Predicted unstable')
    white_patch = mpatches.Patch(facecolor='none', edgecolor='k', linewidth=2, label='Predicted stable')
    figure.legend(handles=[green_patch, black_patch, white_patch], loc='upper right', bbox_to_anchor=(1, 1), fontsize=12)


    plt.tight_layout(rect=[0, 0, 0.9, 0.95])
    plt.suptitle(f'Random forest classifier predicted ASD stability results at {condition}', fontsize=18)
    save_directory = '/home/lero/idrive/cmac/DDMAP/Stability studies/Model_results/Plots'
    plot_filename = os.path.join(save_directory, f'RF_results_heatmap_@_{condition}.png')
    plt.savefig(plot_filename)
    plt.show()
